In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [3]:
# 1. LOAD & CLEAN DATA

df = pd.read_csv("respondent_data/respondent_propensity_training.csv")

# Remove duplicate entries
df = df.drop_duplicates()

# Handle missing values

# For numerical columns, fill missing values with the median
num_cols = df.select_dtypes(include=["number"]).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# For categorical columns: fill missing slots with the mode (most frequent value)
cat_cols = df.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# 2. CREATE TARGET VARIABLE
# target = 1 if survey completion rate (completed/clicked) >= 15%

df["completion_rate"] = (
    df["no_surveys_completed"] /
    df["no_surveys_clicked"]
)

df["propensity_target"] = np.where(
    df["completion_rate"] >= 0.15,
    1,
    0
)
print(df.head())

   user_id loc_india date_of_join date_of_last_invite  \
0        1       Yes     15/08/25            09/05/26   
1        2       Yes     30/07/25            09/05/26   
2        3       Yes     16/03/26            18/05/26   
3        4       Yes     21/05/26            02/05/26   
4        5       Yes     24/02/26            18/05/26   

  date_of_last_participation date_of_last_completion  no_surveys_clicked  \
0                   03/03/26                21/02/26                 146   
1                   02/03/26                13/02/26                  64   
2                   19/04/26                19/04/26                  65   
3                   26/12/25                13/12/25                  26   
4                   09/02/26                30/01/26                  28   

   no_surveys_completed  min_incentive_threshold  max_incentive_threshold  \
0                    28                     2.05                     3.80   
1                     8                     1.

In [4]:
# 3. DATE FEATURE ENGINEERING

today = pd.Timestamp.today().normalize()

date_cols = [
    "date_of_join",
    "date_of_last_invite",
    "date_of_last_participation",
    "date_of_last_completion"
]

for col in date_cols:

    df[col] = pd.to_datetime(df[col],format="mixed", errors="coerce")

    df[f"days_since_{col.replace('date_of_', '')}"] = (
        today - df[col]
    ).dt.days

In [5]:
# 4. SELECT MODEL FEATURES

features = [
    "age",
    "gender",
    "no_surveys_clicked",
    "no_surveys_completed",
    "min_incentive_threshold",
    "max_incentive_threshold",
    "min_survey_length",
    "max_survey_length",
    "days_since_join",
    "days_since_last_invite",
    "days_since_last_participation",
    "days_since_last_completion"
]
X = df[features]
y = df["propensity_target"]

In [6]:
# 5. TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
# 6. PREPROCESSING

categorical_features = [
    "gender"
]

numeric_features = [
    "age",
    "no_surveys_clicked",
    "no_surveys_completed",
    "min_incentive_threshold",
    "max_incentive_threshold",
    "min_survey_length",
    "max_survey_length",
    "days_since_join",
    "days_since_last_invite",
    "days_since_last_participation",
    "days_since_last_completion"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(drop="first"),
            categorical_features
        )
    ]
)

In [8]:
# 7. LOGISTIC REGRESSION MODEL

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'no_surveys_clicked',
                                                   'no_surveys_completed',
                                                   'min_incentive_threshold',
                                                   'max_incentive_threshold',
                                                   'min_survey_length',
                                                   'max_survey_length',
                                                   'days_since_join',
                                                   'days_since_last_invite',
                                                   'days_since_last_participation',
                                                   'days_since_last_completion']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['gender'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [9]:
# 8. EVALUATE MODEL

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

test_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(
    y_test,
    test_prob
)

print("\nTRAIN ACCURACY")
print(train_accuracy)

print("\nTEST ACCURACY")
print(test_accuracy)

print("\nROC AUC")
print(roc_auc)

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, test_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, test_pred))


TRAIN ACCURACY
0.9950125

TEST ACCURACY
0.995275

ROC AUC
0.999966306303958

CONFUSION MATRIX
[[62825    49]
 [  518 56608]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     62874
           1       1.00      0.99      1.00     57126

    accuracy                           1.00    120000
   macro avg       1.00      1.00      1.00    120000
weighted avg       1.00      1.00      1.00    120000



In [10]:
# 9. GENERATE PROPENSITY SCORES

df["propensity_score"] = model.predict_proba(
    X
)[:, 1]

In [11]:
# 10. RANK RESPONDENTS

df["rank"] = (
    df["propensity_score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

In [12]:
# 11. CREATE DECILES

df["decile"] = pd.qcut(
    df["propensity_score"],
    10,
    duplicates='drop'
)

In [13]:
# 12. SORT HIGHEST PROPENSITY FIRST

df_ranked = df.sort_values(
    by="propensity_score",
    ascending=False
)

In [14]:
# 13. SAVE RESULTS

df_ranked.to_csv(
    "respondent_data/respondent_propensity_scored.csv",
    index=False
)

print(
    "\nFile saved as respondent_propensity_scored.csv"
)


File saved as respondent_propensity_scored.csv


In [26]:
# 14. TOP RESPONDENTS

print(
    df_ranked[
        [
            "user_id",
            "propensity_score",
            "rank",
            "decile"
        ]
    ].head(20)
)

        user_id  propensity_score   rank            decile
388723   388724               1.0  79206  (0.9999999, 1.0]
107028   107029               1.0  21839  (0.9999999, 1.0]
343616   343617               1.0  70057  (0.9999999, 1.0]
107022   107023               1.0  21838  (0.9999999, 1.0]
343614   343615               1.0  70056  (0.9999999, 1.0]
343613   343614               1.0  70055  (0.9999999, 1.0]
343612   343613               1.0  70054  (0.9999999, 1.0]
343604   343605               1.0  70053  (0.9999999, 1.0]
343603   343604               1.0  70052  (0.9999999, 1.0]
343618   343619               1.0  70059  (0.9999999, 1.0]
343600   343601               1.0  70051  (0.9999999, 1.0]
343599   343600               1.0  70050  (0.9999999, 1.0]
343595   343596               1.0  70049  (0.9999999, 1.0]
343583   343584               1.0  70048  (0.9999999, 1.0]
343582   343583               1.0  70047  (0.9999999, 1.0]
343581   343582               1.0  70046  (0.9999999, 1.